## CI/CD and Continuous Training (CT)

CI/CD (Continuous Integration / Continuous Delivery) is the practice of automating
the build, test, and deployment pipeline for software.
Continuous Training extends this to ML systems: models are automatically retrained
and re-evaluated when new data arrives or performance degrades, without manual intervention.

## How it actually works

**Standard CI/CD (software):**
Developer pushes code → automated tests run → if tests pass, code is packaged → deployed to staging → deployed to production.
The goal: no manual steps between a code commit and a running service.

**CI/CD for ML (MLOps) adds a model dimension:**
Changes to training code, hyperparameters, or data pipelines all trigger the pipeline.
The pipeline trains a new model, evaluates it against a quality threshold, and only promotes it if it passes.
If a model does not beat the incumbent on key metrics, the deployment is blocked automatically.

**Continuous Training (CT) — the part most software engineers miss:**
Even if your code never changes, your model can degrade.
Real-world data distributions shift over time.
CT means you have a scheduled or event-triggered pipeline that:
1. Ingests fresh training data (e.g. the last 90 days of labeled examples)
2. Retrains the model from scratch (or fine-tunes it)
3. Runs the full evaluation suite
4. If evaluation passes, promotes the new model to production automatically
5. Logs the promotion event with full metadata: data snapshot, metrics, git commit hash

**Triggers for CT:**
- Scheduled: retrain every 7 days regardless of drift
- Data-triggered: retrain when new labeled data volume exceeds a threshold
- Performance-triggered: retrain when live model accuracy drops below a baseline
- Drift-triggered: retrain when input feature distributions shift beyond a threshold

**The three pipelines in a mature ML system:**
1. Data pipeline — validates, cleans, features-engineers incoming data
2. Training pipeline — trains, evaluates, registers model artifacts
3. Serving pipeline — loads the registered model and handles prediction requests
CT orchestrates all three automatically.

**Tooling commonly used:**
GitHub Actions / Azure DevOps for CI/CD.
MLflow, Azure ML, or Vertex AI for experiment tracking and model registry.
Kubeflow Pipelines, Apache Airflow, or Prefect for pipeline orchestration.
The pipeline is code — version-controlled, testable, repeatable.

## Where it breaks or gets dangerous

**1. Promoting without evaluation gates:**
Treating model promotion like a software deployment — code passes unit tests, ship it.
Models can pass code tests and still produce degraded predictions on new data.
Every CT pipeline must enforce a metric threshold before promotion.
No threshold = no protection against quietly shipping a worse model.

**2. Training on stale or contaminated data:**
CT runs on whatever data the data pipeline feeds it.
If data quality degrades upstream — labeling errors, schema changes, upstream API changes —
the CT pipeline will happily train and promote a model on bad data.
Data validation must sit before training, not after.

**3. No model versioning or artifact storage:**
CT runs daily. New model replaces old model. Something breaks at 2am.
You need to roll back. But you've overwritten the previous artifact.
Every trained model must be stored with its full metadata: dataset version, git hash, metrics, timestamp.
You must be able to re-serve any previous version within minutes.

**4. Silent retraining of models with regulatory implications:**
In finance, healthcare, and regulated industries — you cannot retrain and redeploy a model silently.
A model used in credit scoring or financial reporting may require audit evidence
of what training data was used, by whom, and when.
CT in these contexts requires explicit approval gates before promotion, not just automated metric checks.

**5. Divergent training and serving environments:**
CI/CD ensures code is tested in a staging environment before production.
ML teams often skip this — they train in a Jupyter notebook and deploy to production directly.
The training environment must mirror the serving environment precisely.
Otherwise the model that passes evaluation may behave differently once served.

In [ ]:
# CI/CD and Continuous Training — conceptual pipeline simulation

import random
import hashlib
import datetime

class ModelRegistry:
    """Stores trained model artifacts. In production: MLflow, Azure ML, W&B."""

    def __init__(self):
        self.models = []
        self.production_version = None

    def register(self, model):
        self.models.append(model)
        print(f"  Registered: {model['version']} | accuracy={model['accuracy']:.3f} | data_hash={model['data_hash'][:8]}")

    def promote(self, version):
        self.production_version = version
        print(f"  PROMOTED to production: {version}")

    def get_production(self):
        return next((m for m in self.models if m["version"] == self.production_version), None)


def data_validation(data_batch):
    """Gate 1: validate data before training."""
    null_rate = sum(1 for x in data_batch if x is None) / len(data_batch)
    if null_rate > 0.05:
        raise ValueError(f"Data validation failed: null rate {null_rate:.1%} exceeds 5% threshold")
    return True


def train_model(data_batch, version):
    """Simulate model training — returns accuracy and a data fingerprint."""
    data_str = str([x for x in data_batch if x is not None])
    data_hash = hashlib.md5(data_str.encode()).hexdigest()
    # Simulate variable accuracy based on data batch
    accuracy = 0.80 + random.uniform(0, 0.15)
    return {"version": version, "accuracy": accuracy, "data_hash": data_hash,
            "trained_at": datetime.datetime.now().isoformat()}


def evaluation_gate(model, threshold=0.87):
    """Gate 2: block promotion if accuracy falls below threshold."""
    if model["accuracy"] < threshold:
        raise ValueError(f"Evaluation gate failed: {model['accuracy']:.3f} < threshold {threshold}")
    return True


def continuous_training_pipeline(data_batches, registry, threshold=0.87):
    """Simulate CT runs: validate data, train, evaluate, conditionally promote."""
    print(f"Continuous Training Pipeline | threshold={threshold}")
    print("-" * 60)

    for i, batch in enumerate(data_batches):
        version = f"v1.{i+1}"
        print(f"\nRun {i+1}: training {version}")

        try:
            data_validation(batch)
            print(f"  Data validation: PASSED ({len([x for x in batch if x is not None])} valid rows)")
        except ValueError as e:
            print(f"  Data validation: FAILED — {e}")
            print(f"  Aborting run. Previous production model remains active.")
            continue

        model = train_model(batch, version)
        registry.register(model)

        try:
            evaluation_gate(model, threshold)
            print(f"  Evaluation gate: PASSED")
            registry.promote(version)
        except ValueError as e:
            print(f"  Evaluation gate: BLOCKED — {e}")
            prod = registry.get_production()
            if prod:
                print(f"  Current production ({prod['version']}, {prod['accuracy']:.3f}) remains unchanged.")
            else:
                print(f"  No production model exists yet.")

    print("\n" + "=" * 60)
    prod = registry.get_production()
    if prod:
        print(f"Final production model: {prod['version']} | accuracy={prod['accuracy']:.3f}")
    else:
        print("No model reached production threshold.")


random.seed(42)

batches = [
    [random.random() for _ in range(200)],          # good batch
    [random.random() if i % 8 != 0 else None        # batch with null contamination
     for i in range(200)],
    [random.random() for _ in range(200)],          # good batch
    [random.random() for _ in range(200)],          # good batch
]

registry = ModelRegistry()
continuous_training_pipeline(batches, registry, threshold=0.87)

## My D365 analogy

CI/CD in ML maps cleanly to D365 release management.

In D365 implementations, every code change — X++ customisation, data entity change,
workflow modification — goes through a structured pipeline:
development → build → automated testing → UAT → production.
No one deploys directly to production. The pipeline enforces it.

The evaluation gate in CT is the equivalent of UAT sign-off.
A D365 release doesn't go live if UAT fails.
A retrained model doesn't go live if accuracy drops below threshold.
Same discipline, different domain.

Continuous Training maps to the period-end data refresh cycle in D365.
At period-end, every subledger is reconciled, financial dimensions are validated,
and closing journals are posted in a defined sequence.
The process runs on a schedule, is fully auditable, and has hard gates at each step.
A step that fails blocks the next one — not silently, but with a documented error.

CT pipelines work the same way.
Data validation fails — pipeline stops, production model unchanged.
Evaluation gate fails — pipeline stops, production model unchanged.
Every run is logged. Every promotion is traceable.

The D365 principle: never post to the general ledger without a validated subledger.
The ML equivalent: never promote to production without a validated evaluation result.

## LinkedIn post idea

In D365, nobody pushes a customisation directly to production.
It goes through build pipelines, automated tests, UAT, sign-off.
The discipline is so ingrained that doing it any other way feels reckless.

I spent weeks building my first ML training pipeline,
and then I realised I had no evaluation gate before promotion.
I was training and deploying. No threshold. No block. Just shipping.

That is the equivalent of applying a customisation to production
without running a single test.

Continuous Training is not just about keeping models fresh.
It is about applying the same release discipline to models
that enterprise teams apply to code.

Data validation before training.
Metric threshold before promotion.
Every run logged. Every artifact versioned.

The tooling is different. The principle is not.